In [2]:
import numpy as np

def bootstrap_indices(n):
    return np.random.choice(list(range(n)),size = n,replace=True)

In [3]:
idx = bootstrap_indices(5)
print(idx)
print(len(idx))

[4 0 4 1 2]
5


In [5]:
def bootstrap_sample(X, y):
    idx = bootstrap_indices(len(X))
    X_boot = X[idx]
    y_boot = y[idx]
    return X_boot, y_boot

In [6]:
X = np.array([[10], [20], [30], [40], [50]])
y = np.array([1, 2, 3, 4, 5])

X_boot, y_boot = bootstrap_sample(X, y)

print(X_boot.ravel())
print(y_boot)

[50 10 40 10 40]
[5 1 4 1 4]


In [7]:
def rss(y):
    y_mean = np.mean(y)
    total = 0
    for i in range(y.size):
        total += (y[i] - y_mean) ** 2
    return total

def rss_reduction(y_parent, y_left, y_right):
    return rss(y_parent) - rss(y_left) - rss(y_right)

def create_thresholds(x):
    x = np.sort(np.unique(x))
    y = []
    for i in range(len(x)-1):
        y.append((x[i] + x[i+1])/2)
    return y

def split(x, y, threshold):
    y_left = []
    y_right = []
    for i in range(len(y)):
        if(x[i] <= threshold):
            y_left.append(y[i])
        else:
            y_right.append(y[i])
    return np.array(y_left), np.array(y_right)

def best_split_for_feature_regression(x, y):
    thresholds = create_thresholds(x)

    best_threshold = None
    best_reduction = -np.inf

    for threshold in thresholds:
        y_left, y_right = split(x, y, threshold)
        rss_val = rss_reduction(y,y_left,y_right)
        if(rss_val > best_reduction):
            best_reduction = rss_val
            best_threshold = threshold
        
    return best_threshold, best_reduction

def best_split_regression(X, y):
    best_feature = None
    best_threshold = None
    best_reduction = -np.inf

    for j in range(X.shape[1]):
        threshold,reduction = best_split_for_feature_regression(X[:, j], y)
        if(reduction > best_reduction):
            best_feature = j
            best_threshold = threshold
            best_reduction = reduction
        
    return best_feature, best_threshold, best_reduction

def split_data(X, y, feature, threshold):
    left_mask = X[:, feature] <= threshold
    right_mask = X[:, feature] > threshold

    X_left = X[left_mask]
    y_left = y[left_mask]

    X_right = X[right_mask]
    y_right = y[right_mask]

    return X_left, y_left, X_right, y_right

def build_regression_tree(X, y,depth=0,max_depth=3,min_samples_split=2):
    if(np.all(y == y[0])):
        return {"leaf": True, "prediction" : float(y[0])}
    if(depth >= max_depth or len(y) < min_samples_split):
        return {"leaf": True, "prediction" : np.mean(y)}

    feature, threshold, reduction = best_split_regression(X, y)
    if reduction <= 0:
        return {"leaf": True, "prediction": np.mean(y)}
    
    X_left, y_left, X_right, y_right = split_data(X, y, feature, threshold)
    left_tree = build_regression_tree(X_left,y_left,depth+1,max_depth,min_samples_split)
    right_tree = build_regression_tree(X_right,y_right,depth+1,max_depth,min_samples_split)

    return {
        "leaf": False,
        "feature": feature,
        "threshold": threshold,
        "left": left_tree,
        "right": right_tree
    }

In [8]:
X = np.array([[1],[2],[3],[4],[5],[6]])
y = np.array([1.0, 2.0, 2.5, 4.0, 5.0, 6.0])

X_boot, y_boot = bootstrap_sample(X, y)
tree = build_regression_tree(X_boot,y_boot,max_depth=3)

print(tree)

{'leaf': False, 'feature': 0, 'threshold': np.float64(3.0), 'left': {'leaf': False, 'feature': 0, 'threshold': np.float64(1.5), 'left': {'leaf': True, 'prediction': 1.0}, 'right': {'leaf': True, 'prediction': 2.0}}, 'right': {'leaf': True, 'prediction': 4.0}}


In [10]:
def fit_bagged_trees(X, y, B=10, max_depth=3):
    trees = []

    for _ in range(B):
        X_boot, y_boot = bootstrap_sample(X, y)
        tree = build_regression_tree(X_boot,y_boot,max_depth=max_depth)
        trees.append(tree)

    return trees

In [12]:
def predict_one_regression(x, tree):
    if tree["leaf"]:
        return tree["prediction"]

    if x[tree["feature"]] <= tree["threshold"]:
        return predict_one_regression(x, tree["left"])
    else:
        return predict_one_regression(x, tree["right"])

def predict_regression(X, tree):
    predictions = []

    for row in X:
        predictions.append(predict_one_regression(row,tree))

    return np.array(predictions)


def predict_bagged(X, trees):
    all_predictions = []

    for tree in trees:
        all_predictions.append(predict_regression(X, tree))
    
    all_predictions = np.array(all_predictions)
    
    return np.mean(all_predictions, axis=0)

In [13]:
X = np.array([[1],[2],[3],[4],[5],[6]])
y = np.array([1.0, 2.0, 2.5, 4.0, 5.0, 6.0])

trees = fit_bagged_trees(X,y,B=20,max_depth=3)
pred = predict_bagged(X, trees)

print("Predictions:", pred)
print("Targets:    ", y)

Predictions: [1.25  1.8   2.475 3.425 4.825 5.425]
Targets:     [1.  2.  2.5 4.  5.  6. ]


In [14]:
x_test = np.array([[4.5]])

tree_predictions = np.array([predict_regression(x_test, tree)[0] for tree in trees])

print("Individual tree predictions:")
print(tree_predictions)

print("Mean:", np.mean(tree_predictions))
print("Std:", np.std(tree_predictions))

Individual tree predictions:
[5.  5.  5.  4.  2.5 4.  4.  2.5 5.  4.  2.5 5.  2.5 4.  4.  5.  4.  4.
 4.  2.5]
Mean: 3.925
Std: 0.9256754290786808


In [15]:
x_test = np.array([[4.5]])

single_preds = []
bagged_preds = []

for _ in range(100):
    X_boot, y_boot = bootstrap_sample(X, y)
    tree = build_regression_tree(X_boot,y_boot,max_depth=3)

    single_pred = predict_regression(x_test, tree)[0]
    single_preds.append(single_pred)

    trees = fit_bagged_trees(X,y,B=20,max_depth=3)
    
    bagged_pred = predict_bagged(x_test, trees)[0]
    bagged_preds.append(bagged_pred)

print("Single tree std:", np.std(single_preds))
print("Bagged model std:", np.std(bagged_preds))

print("Single tree mean:", np.mean(single_preds))
print("Bagged model mean:", np.mean(bagged_preds))

Single tree std: 0.7428828979051812
Bagged model std: 0.14451881365413988
Single tree mean: 4.025
Bagged model mean: 4.16925


In [16]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_squared_error

ROOT = Path.cwd().parents[1]
DATA_DIR = ROOT / "data"

df = pd.read_csv(DATA_DIR / "duolingo_flagship_v5.csv")
split_users = pd.read_csv(DATA_DIR / "split_users.csv")

cv_users = split_users.loc[split_users["split"] == "cv","user_id"]
cv_df = df[df["user_id"].isin(cv_users)].copy()

features = [
    "lag_days",
    "history_seen",
    "history_correct",
    "history_accuracy",
    "lag_days_log"
]

X = cv_df[features]
y = cv_df["p_recall"]
groups = cv_df["user_id"]

gkf = GroupKFold(n_splits=5)

print(len(cv_df), cv_df["user_id"].nunique())

14438 2125


In [17]:
fold_rmses = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups),start=1):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42),
        n_estimators=100,
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))

    fold_rmses.append(rmse)
    print(f"Fold {fold}: {rmse:.6f}")

print()
print(f"Mean RMSE: {np.mean(fold_rmses):.6f}")
print(f"Std RMSE:  {np.std(fold_rmses):.6f}")

Fold 1: 0.307029
Fold 2: 0.302155
Fold 3: 0.288708
Fold 4: 0.288679
Fold 5: 0.310004

Mean RMSE: 0.299315
Std RMSE:  0.009027


In [18]:
tree_counts = [1, 5, 10, 20, 50, 100, 200]
results = []

for B in tree_counts:
    fold_rmses = []

    for train_idx, val_idx in gkf.split(X, y, groups):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]

        model = BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42),
            n_estimators=B,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        pred = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, pred))
        fold_rmses.append(rmse)

    results.append({
        "n_estimators": B,
        "mean_rmse": np.mean(fold_rmses),
        "std_rmse": np.std(fold_rmses)
    })

pd.DataFrame(results)

,n_estimators,mean_rmse,std_rmse
0,1,0.370501,0.008133
1,5,0.314956,0.009051
2,10,0.307000,0.008200
3,20,0.302867,0.008401
4,50,0.300229,0.008756
5,100,0.299315,0.009027
6,200,0.299097,0.009136


## Part 13 — Bootstrap and Bagging

Implemented bootstrap sampling and bagging from scratch.

Key ideas:
- Bootstrap sampling draws `n` observations from a dataset of size `n` with replacement.
- Each bootstrap sample trains a different regression tree.
- Bagging averages tree predictions:

$$
\hat{f}_{\mathrm{bag}}(x)
=
\frac{1}{B}
\sum_{b=1}^{B}\hat{f}_b(x)
$$
- Averaging unstable trees reduces prediction variance.

Toy experiment:
- Single-tree prediction std ≈ 0.743
- Bagged-model prediction std ≈ 0.145

Flagship GroupKFold results:
- 1 tree: RMSE ≈ 0.3705
- 5 trees: RMSE ≈ 0.3150
- 20 trees: RMSE ≈ 0.3029
- 100 trees: RMSE ≈ 0.2993
- 200 trees: RMSE ≈ 0.2991

Interpretation:
Bagging substantially reduces the variance of an unrestricted decision tree, but performance plateaus because additional trees are correlated and increasingly redundant.

Bagging alone does not match the regularized single tree (~0.2739 RMSE) or Ridge (~0.2735 RMSE).